In [1]:
import os
os.environ["HF_HOME"] = "/scratch/heli"
os.environ["HF_DATASETS_CACHE"] = "/scratch/heli"

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"

In [4]:
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [5]:
model.save_pretrained('/scratch/heli/models_from_hf/llama_1b_inst')

In [6]:
tok = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [7]:
tok.save_pretrained('/scratch/heli/models_from_hf/llama_1b_inst/')

('/scratch/heli/models_from_hf/llama_1b_inst/tokenizer_config.json',
 '/scratch/heli/models_from_hf/llama_1b_inst/special_tokens_map.json',
 '/scratch/heli/models_from_hf/llama_1b_inst/tokenizer.json')

In [18]:
import pandas as pd
from datasets import load_dataset, load_from_disk

In [39]:
ds = load_from_disk('./ai4bharat/IN22-Conv/')

In [40]:
ds = ds['test']
ds

Dataset({
    features: ['doc_id', 'sent_id', 'topic', 'domain', 'prompt', 'scenario', 'speaker', 'turn', 'asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'eng_Latn', 'gom_Deva', 'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'mai_Deva', 'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya', 'pan_Guru', 'san_Deva', 'sat_Olck', 'snd_Deva', 'tam_Taml', 'tel_Telu', 'urd_Arab'],
    num_rows: 1503
})

In [36]:
ds['eng_Latn'][0]

"Mom, let's go for a movie tomorrow."

In [37]:
ds['guj_Gujr'][0]

'મમ્મી, ચાલો આપણે આવતીકાલે ફિલ્મ જોવા જઇએ.'

In [1]:
from datasets import load_from_disk

In [42]:
ds_conv = load_from_disk('../Layerwise/Data/ai4bharat/IN22-Conv/')['test']
ds_gen = load_from_disk('../Layerwise/Data/ai4bharat/IN22-Gen/')['test']
ds_flores = load_from_disk('../Layerwise/Data/openlanguagedata/flores_plus/',  "default")

In [43]:
ds_gen, ds_conv

(Dataset({
     features: ['context', 'source', 'url', 'domain', 'num_words', 'bucket', 'asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'eng_Latn', 'gom_Deva', 'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'mai_Deva', 'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya', 'pan_Guru', 'san_Deva', 'sat_Olck', 'snd_Deva', 'tam_Taml', 'tel_Telu', 'urd_Arab'],
     num_rows: 1024
 }),
 Dataset({
     features: ['doc_id', 'sent_id', 'topic', 'domain', 'prompt', 'scenario', 'speaker', 'turn', 'asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'eng_Latn', 'gom_Deva', 'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'mai_Deva', 'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya', 'pan_Guru', 'san_Deva', 'sat_Olck', 'snd_Deva', 'tam_Taml', 'tel_Telu', 'urd_Arab'],
     num_rows: 1503
 }))

In [44]:
ds_flores

DatasetDict({
    dev: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 223328
    })
    devtest: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 220616
    })
})

In [45]:
ds = ds_flores
tgt_langs = ds["devtest"]["iso_639_3"]
src_langs = ds["devtest"]["iso_639_3"]

eng_ds = ds['devtest'].filter(lambda x: x['iso_639_3']=='eng')
hin_ds = ds['devtest'].filter(lambda x: x['iso_639_3']=='hin')


eng_hin = []
hin_eng = []

eng_src = []
hin_src = []

# prompt = """Translate the following text from {SOURCE_LANGUAGE} to {TARGET_LANGUAGE}.
# Ensure the translation is accurate, fluent, and faithful to the original meaning.
# Do not add, remove, or change any information.

# Text:
# {SOURCE_TEXT}"""

prompt = """Translate following {SOURCE_LANGUAGE} dialogue to {TARGET_LANGUAGE}.\nSource Sentence:\n{SOURCE_TEXT}\nTarget:\n{TARGET_TEXT}"""

eng_src.extend([*eng_ds['text']])#, *in22_gen['test']['eng_Latn'], *in22_conv['test']['eng_Latn']])
hin_src.extend([*hin_ds['text']])#, *in22_gen['test']['hin_Deva'], *in22_conv['test']['hin_Deva']])

eng_hin.extend([prompt.format(SOURCE_LANGUAGE="English", TARGET_LANGUAGE="Hindi", SOURCE_TEXT=s.strip(), TARGET_TEXT=t.strip()) for s,t in zip(eng_ds['text'], hin_ds['text'])])
eng_hin.extend([prompt.format(SOURCE_LANGUAGE="English", TARGET_LANGUAGE="Hindi", SOURCE_TEXT=s.strip(), TARGET_TEXT=t.strip()) for s,t in zip(ds_gen['eng_Latn'], ds_gen['hin_Deva'])])
eng_hin.extend([prompt.format(SOURCE_LANGUAGE="English", TARGET_LANGUAGE="Hindi", SOURCE_TEXT=s.strip(), TARGET_TEXT=t.strip()) for s,t in zip(ds_conv['eng_Latn'], ds_conv['hin_Deva'])])

# hin_eng.extend([prompt.format(SOURCE_LANGUAGE="Hindi", TARGET_LANGUAGE="English", SOURCE_TEXT=s.strip()) for s in hin_ds['text']])

Filter:   0%|          | 0/220616 [00:00<?, ? examples/s]

Filter:   0%|          | 0/220616 [00:00<?, ? examples/s]

In [51]:
print(eng_hin[100])

Translate following English dialogue to Hindi.
Source Sentence:
In one year's time, an infected person may infect 10 to 15 close contacts.
Target:
एक साल के दौरान, एक संक्रमित व्यक्ति 10 से 15 करीबी संपर्कों को संक्रमित कर सकता है.


In [47]:
len(eng_ds)

1012

In [48]:
len(hin_ds)

1012

In [49]:
len(eng_hin)

3539

In [50]:
ds_flores['devtest'][0]

{'id': 0,
 'iso_639_3': 'ace',
 'iso_15924': 'Arab',
 'glottocode': 'achi1257',
 'variant': '',
 'text': '"کامو جينو نا تيكويه عمو ٤ بولن ڽڠ هانا ديابيتيس ڽڠ اوايجيه ساکيت ديابيتيس،" غتامه لى غوبڽن.',
 'url': 'https://en.wikinews.org/wiki/Toronto_team-led_research_on_Type_1_Diabetes_%27groundbreaking%27',
 'domain': 'wikinews',
 'topic': 'disease, research, canada',
 'has_image': 'no',
 'has_hyperlink': 'yes',
 'last_updated': '1.0',
 'split': 'devtest'}

In [53]:
import pandas as pd

In [54]:
df = pd.DataFrame({
    "prompt": eng_hin
})

In [56]:
df.to_csv('./train_dataset.tsv', sep='\t')

In [57]:
ds = load_from_disk('../Layerwise/Data/Opus_Test_Data_en_hi/')

In [61]:
ds = ds.select(range(100))

In [67]:
eng_hin = []
hin_eng = []

prompt = """Translate following {SOURCE_LANGUAGE} dialogue to {TARGET_LANGUAGE}.\nSource Sentence:\n{SOURCE_TEXT}\nTarget:\n{TARGET_TEXT}"""

eng_hin.extend([prompt.format(SOURCE_LANGUAGE="English", TARGET_LANGUAGE="Hindi", SOURCE_TEXT=s.strip(), TARGET_TEXT=t.strip()) for s,t in zip(ds['source'], ds['target'])])

In [70]:
print(eng_hin[0])

Translate following English dialogue to Hindi.
Source Sentence:
Give shots of injections or pills, but he must be alright soon.
Target:
सुई लगाओ या गोली खिलाओ लेकिन इसे जल्दी से ठीक करो.


In [71]:
df = pd.DataFrame({
    "prompt": eng_hin
})

df.to_csv('./valid_dataset.tsv', sep='\t')

In [ ]:
CUDA_VISIBLE_DEVICES=1 python3 llama.py /scratch/heli/models_from_hf/llama_1b_inst c4 --seed 42 --sparsity 0.5 --save /scratch/heli/llama_3B_INST_sparseGPT_pruned_structured_50_4_8 --prunen 4 --prunem 8

In [72]:
"""llama_3B_INST_sparseGPT_pruned_structured_25_1_4
llama_3B_INST_sparseGPT_pruned_structured_25_2_8
llama_3B_INST_sparseGPT_pruned_structured_50_4_8
llama_3B_INST_sparseGPT_pruned_unstructured_25
llama_3B_INST_sparseGPT_pruned_unstructured_75""".split("\n")

['llama_3B_INST_sparseGPT_pruned_structured_25_1_4',
 'llama_3B_INST_sparseGPT_pruned_structured_25_2_8',
 'llama_3B_INST_sparseGPT_pruned_structured_50_4_8',
 'llama_3B_INST_sparseGPT_pruned_unstructured_25',
 'llama_3B_INST_sparseGPT_pruned_unstructured_75']

In [1]:
from huggingface_hub import snapshot_download

In [2]:
snapshot_download(
    repo_id="Yuvrajsinh0409/Llama_1B_inst_in22_flores_ft",
    repo_type="model",
    cache_dir="/scratch/heli/",
    local_dir="/scratch/heli/"
)

Fetching 41 files:   0%|          | 0/41 [00:00<?, ?it/s]

'/scratch/heli'

In [ ]:
/scratch/heli/Llama_FT/merged-checkpoint-280/
/home2/abhinav.pm/yuvraj/Pruning/Layerwise/Data/Opus_Test_Data_en_hi/
'./Inference/Inference_vllm_Llama_3.2_1B_Base.tsv'

python3 llama.py Llama_FT/merged-checkpoint-280/ c4 --seed 42 --sparsity 0.25 --prunen 1 --prunem 4 --save ./llama_1B_INST_FT_sparseGPT_pruned_structured_25_1_4

In [45]:
import random
from tqdm.auto import tqdm
def get_c4(nsamples, seed, seqlen, model, tokenizer):
    traindata = load_dataset(
       "csv",
    data_files='/home/vish/yuvraj/Layerwise/Data/train_dataset.tsv',
    delimiter="\t",
    split="train"
    )
    valdata = load_dataset(
        "csv",
    data_files='/home/vish/yuvraj/Layerwise/Data/valid_dataset.tsv',
    delimiter="\t",
    split="train"
    )

    random.seed(seed)
    trainloader = []
    for _ in tqdm(range(nsamples)):
        while True:
            i = random.randint(0, len(traindata) - 1)
            trainenc = tokenizer(traindata[i]['prompt'], return_tensors='pt')
            print(trainenc.input_ids.shape[1], seqlen)
            if trainenc.input_ids.shape[1] > seqlen:
                break
        i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
        j = i + seqlen
        inp = trainenc.input_ids[:, i:j]
        tar = inp.clone()
        tar[:, :-1] = -100
        trainloader.append((inp, tar))

    valenc = tokenizer(' '.join(valdata[:1100]['prompt']), return_tensors='pt')
    valenc = valenc.input_ids[:, :(256 * seqlen)]

    class TokenizerWrapper:
        def __init__(self, input_ids):
            self.input_ids = input_ids
    valenc = TokenizerWrapper(valenc)

    return trainloader, valenc


def get_wikitext2(nsamples, seed, seqlen, model, tokenizer):
    
    traindata = load_dataset(
       "csv",
    data_files='/home/vish/yuvraj/Layerwise/Data/train_dataset.tsv',
    delimiter="\t",
    split="train"
    )
    testdata = load_dataset(
        "csv",
    data_files='/home/vish/yuvraj/Layerwise/Data/valid_dataset.tsv',
    delimiter="\t",
    split="train"
    )

    trainenc = tokenizer(" ".join(traindata['prompt']), return_tensors='pt')
    testenc = tokenizer("\n\n".join(testdata['prompt']), return_tensors='pt')

    random.seed(seed)
    trainloader = []
    for _ in range(nsamples):
        i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
        j = i + seqlen
        inp = trainenc.input_ids[:, i:j]
        tar = inp.clone()
        tar[:, :-1] = -100
        trainloader.append((inp, tar))
    return trainloader, testenc

In [17]:
from datasets import load_from_disk, load_dataset
from transformers import AutoTokenizer

In [35]:
def get_tokenizer(model):
    if "llama" in model.lower():
        tokenizer = AutoTokenizer.from_pretrained(model, use_fast=False)
        # fix for transformer 4.28.0.dev0 compatibility
        if tokenizer.bos_token_id != 1 or tokenizer.eos_token_id != 2:
            try:
                tokenizer.bos_token_id = 1
                tokenizer.eos_token_id = 2
            except AttributeError:
                pass
    else:
        tokenizer = AutoTokenizer.from_pretrained(model, use_fast=False)
    return tokenizer

tok = get_tokenizer('./Llama_FT/merged-checkpoint-280/')

In [18]:
tokenizer = AutoTokenizer.from_pretrained('./Llama_FT/merged-checkpoint-280/')

In [11]:
ds = load_dataset("csv",
    data_files='./train_dataset.tsv',
    delimiter="\t",
    split="train")


Generating train split: 0 examples [00:00, ? examples/s]

In [13]:
ds[0]["prompt"]

'Translate following English dialogue to Hindi.\nSource Sentence:\n"We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.\nTarget:\nउन्होंने कहा “कि अब हमारे पास 4 महीने उम्र वाले चूहे हैं जिन्हें मधुमेह नहीं है जो मधुमेह के रोगी थे। ”'

In [44]:
d = get_c4(nsamples=128, seed=42, seqlen=150, model=None, tokenizer=tok)

  0%|          | 0/128 [00:00<?, ?it/s]

48 150
91 150
122 150
81 150
190 150
119 150
129 150
63 150
124 150
41 150
62 150
57 150
85 150
38 150
110 150
103 150
76 150
132 150
104 150
134 150
34 150
37 150
166 150
62 150
37 150
34 150
33 150
105 150
69 150
79 150
42 150
73 150
41 150
115 150
35 150
54 150
85 150
53 150
101 150
79 150
90 150
111 150
112 150
72 150
92 150
117 150
71 150
103 150
100 150
97 150
41 150
61 150
95 150
260 150
92 150
38 150
239 150
157 150
128 150
41 150
201 150
74 150
45 150
46 150
61 150
103 150
79 150
66 150
91 150
49 150
118 150
32 150
117 150
140 150
41 150
116 150
84 150
103 150
90 150
70 150
22 150
51 150
92 150
108 150
151 150
65 150
37 150
113 150
59 150
35 150
28 150
104 150
59 150
51 150
84 150
72 150
36 150
66 150
107 150
92 150
282 150
93 150
38 150
94 150
71 150
49 150
140 150
36 150
51 150
52 150
75 150
66 150
39 150
143 150
26 150
146 150
137 150
65 150
146 150
108 150
34 150
45 150
190 150
95 150
46 150
116 150
56 150
206 150
197 150
84 150
37 150
41 150
75 150
189 150
168 150
83 150


In [46]:
d = get_wikitext2(nsamples=128, seed=42, seqlen=2048, model=None, tokenizer=tok)

Token indices sequence length is longer than the specified maximum sequence length for this model (316125 > 131072). Running this sequence through the model will result in indexing errors


In [47]:
d

([(tensor([[100276, 121834, 101782,  ..., 102302, 101795,  35470]]),
   tensor([[ -100,  -100,  -100,  ...,  -100,  -100, 35470]])),
  (tensor([[ 48909,  35470,  48909,  ..., 100303, 101530, 103673]]),
   tensor([[  -100,   -100,   -100,  ...,   -100,   -100, 103673]])),
  (tensor([[103396, 116704, 115874,  ..., 100329,  86133, 110016]]),
   tensor([[  -100,   -100,   -100,  ...,   -100,   -100, 110016]])),
  (tensor([[  1023,   6302,    311,  ...,  35470,  85410, 103552]]),
   tensor([[  -100,   -100,   -100,  ...,   -100,   -100, 103552]])),
  (tensor([[  7318,   8554,    690,  ..., 100460, 102557,  24810]]),
   tensor([[ -100,  -100,  -100,  ...,  -100,  -100, 24810]])),
  (tensor([[105135,  45279, 100358,  ...,   1101,   7263,  15256]]),
   tensor([[ -100,  -100,  -100,  ...,  -100,  -100, 15256]])),
  (tensor([[ 35470,  85410, 100400,  ...,  74958,  10758,    627]]),
   tensor([[-100, -100, -100,  ..., -100, -100,  627]])),
  (tensor([[101201, 100329, 102317,  ..., 100620, 100293,

In [48]:
import shutil

In [49]:
cp = ['/home/vish/tokenizer_config.json', '/home/vish/tokenizer.json', '/home/vish/special_tokens_map.json']

In [56]:
from pathlib import Path
p = Path('./Models/')
models = [i for i in p.glob("llama*")]

In [59]:
for i in models:
    print(i)

Models/llama_1B_INST_FT_sparseGPT_pruned_unstructured_75
Models/llama_1B_INST_FT_sparseGPT_pruned_structured_75_6_8
Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_structured_50_2_4
Models/llama_3B_INST_sparseGPT_pruned_structured_25_2_8
Models/llama_1B_INST_sparseGPT_pruned_structured_75_3_4
Models/llama_3B_INST_sparseGPT_pruned_structured_25_1_4
Models/llama_1B_INST_FT_sparseGPT_pruned_structured_25_2_8
Models/llama_1B_INST_FT_sparseGPT_pruned_structured_75_3_4
Models/llama_3B_INST_sparseGPT_pruned_structured_75_6_8
Models/llama_1B_INST_FT_sparseGPT_pruned_unstructured_50
Models/llama_1B_INST_FT_sparseGPT_pruned_structured_50_2_4
Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_structured_50_4_8
Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_structured_25_2_8
Models/llama_1B_INST_FT_sparseGPT_pruned_unstructured_25
Models/llama_1B_INST_FT_sparseGPT_pruned_structured_25_1_4
Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_unstructured_25
Models/llama_1B_INST_trans_prompt_sp

In [58]:
for i in models:
    for j in cp:
        shutil.copy(j, i)

In [61]:
"""Models/llama_1B_INST_FT_sparseGPT_pruned_unstructured_75
Models/llama_1B_INST_FT_sparseGPT_pruned_structured_75_6_8
Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_structured_50_2_4
Models/llama_3B_INST_sparseGPT_pruned_structured_25_2_8
Models/llama_1B_INST_sparseGPT_pruned_structured_75_3_4
Models/llama_3B_INST_sparseGPT_pruned_structured_25_1_4
Models/llama_1B_INST_FT_sparseGPT_pruned_structured_25_2_8
Models/llama_1B_INST_FT_sparseGPT_pruned_structured_75_3_4
Models/llama_3B_INST_sparseGPT_pruned_structured_75_6_8
Models/llama_1B_INST_FT_sparseGPT_pruned_unstructured_50
Models/llama_1B_INST_FT_sparseGPT_pruned_structured_50_2_4
Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_structured_50_4_8
Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_structured_25_2_8
Models/llama_1B_INST_FT_sparseGPT_pruned_unstructured_25
Models/llama_1B_INST_FT_sparseGPT_pruned_structured_25_1_4
Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_unstructured_25
Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_unstructured_75
Models/llama_3B_INST_sparseGPT_pruned_structured_50_4_8
Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_structured_75_3_4
Models/llama_3B_INST_sparseGPT_pruned_unstructured_75
Models/llama_1B_INST_FT_sparseGPT_pruned_structured_50_4_8
Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_structured_75_6_8
Models/llama_3B_INST_sparseGPT_pruned_unstructured_25
Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_unstructured_50
Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_structured_25_1_4""".split("\n")

['Models/llama_1B_INST_FT_sparseGPT_pruned_unstructured_75',
 'Models/llama_1B_INST_FT_sparseGPT_pruned_structured_75_6_8',
 'Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_structured_50_2_4',
 'Models/llama_3B_INST_sparseGPT_pruned_structured_25_2_8',
 'Models/llama_1B_INST_sparseGPT_pruned_structured_75_3_4',
 'Models/llama_3B_INST_sparseGPT_pruned_structured_25_1_4',
 'Models/llama_1B_INST_FT_sparseGPT_pruned_structured_25_2_8',
 'Models/llama_1B_INST_FT_sparseGPT_pruned_structured_75_3_4',
 'Models/llama_3B_INST_sparseGPT_pruned_structured_75_6_8',
 'Models/llama_1B_INST_FT_sparseGPT_pruned_unstructured_50',
 'Models/llama_1B_INST_FT_sparseGPT_pruned_structured_50_2_4',
 'Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_structured_50_4_8',
 'Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_structured_25_2_8',
 'Models/llama_1B_INST_FT_sparseGPT_pruned_unstructured_25',
 'Models/llama_1B_INST_FT_sparseGPT_pruned_structured_25_1_4',
 'Models/llama_1B_INST_trans_prompt_sparseG

In [62]:
l = ['Models/llama_1B_INST_FT_sparseGPT_pruned_unstructured_75',
 'Models/llama_1B_INST_FT_sparseGPT_pruned_structured_75_6_8',
 'Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_structured_50_2_4',
 'Models/llama_3B_INST_sparseGPT_pruned_structured_25_2_8',
 'Models/llama_1B_INST_sparseGPT_pruned_structured_75_3_4',
 'Models/llama_3B_INST_sparseGPT_pruned_structured_25_1_4',
 'Models/llama_1B_INST_FT_sparseGPT_pruned_structured_25_2_8',
 'Models/llama_1B_INST_FT_sparseGPT_pruned_structured_75_3_4',
 'Models/llama_3B_INST_sparseGPT_pruned_structured_75_6_8',
 'Models/llama_1B_INST_FT_sparseGPT_pruned_unstructured_50',
 'Models/llama_1B_INST_FT_sparseGPT_pruned_structured_50_2_4',
 'Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_structured_50_4_8',
 'Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_structured_25_2_8',
 'Models/llama_1B_INST_FT_sparseGPT_pruned_unstructured_25',
 'Models/llama_1B_INST_FT_sparseGPT_pruned_structured_25_1_4',
 'Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_unstructured_25',
 'Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_unstructured_75',
 'Models/llama_3B_INST_sparseGPT_pruned_structured_50_4_8',
 'Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_structured_75_3_4',
 'Models/llama_3B_INST_sparseGPT_pruned_unstructured_75',
 'Models/llama_1B_INST_FT_sparseGPT_pruned_structured_50_4_8',
 'Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_structured_75_6_8',
 'Models/llama_3B_INST_sparseGPT_pruned_unstructured_25',
 'Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_unstructured_50',
 'Models/llama_1B_INST_trans_prompt_sparseGPT_pruned_structured_25_1_4']

len(l)

25

In [65]:
l = list(Path('./sentence_scores/').glob("*.tsv"))

In [66]:
len(l)

20

In [68]:
import pandas as pd

In [69]:
df = pd.read_csv('./Inference/llama_1B_INST_FT_sparseGPT_pruned_structured_75_3_4.tsv', sep='\t')

In [70]:
df

,Unnamed: 0,Source_Dialogue,Reference,MT
0,0,"Give shots of injections or pills, but he must...",सुई लगाओ या गोली खिलाओ लेकिन इसे जल्दी से ठीक ...,NaN
1,1,"They said, “O Shuaib, we do not understand muc...",और वह लोग कहने लगे ऐ शुएब जो बाते तुम कहते हो ...,.
2,2,- Yeah.,- हाँ.,D.
3,3,If evil befalls him he is perturbed;,"जि उसे तकलीफ़ पहुँचती है तो घबरा उठता है,",NaN
4,4,♪ BE FOREVER BOUND,♪हमेशाके लिएबाध्यहोने,NaN
...,...,...,...,...
1995,1995,Do they not see that Allah enlarges the provis...,क्या उन लोगों ने (इतना भी) ग़ौर नहीं किया कि खु...,NaN
1996,1996,Say: 'Who is the Lord of the heavens and the e...,"कहो, ""आकाशों और धरती का रब कौन है?"" कहो, ""अल्ल...",NaN
1997,1997,-(CELL PHONE RINGING),(सेल फोन बज),D.
1998,1998,"Yeah, smoke it up there, uh, Skippy.","- हाँ, धूंए में उड़ा दो, मेरे शेर.",.


In [71]:
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

In [72]:
df.fillna('-')

,Source_Dialogue,Reference,MT
0,"Give shots of injections or pills, but he must...",सुई लगाओ या गोली खिलाओ लेकिन इसे जल्दी से ठीक ...,-
1,"They said, “O Shuaib, we do not understand muc...",और वह लोग कहने लगे ऐ शुएब जो बाते तुम कहते हो ...,.
2,- Yeah.,- हाँ.,D.
3,If evil befalls him he is perturbed;,"जि उसे तकलीफ़ पहुँचती है तो घबरा उठता है,",-
4,♪ BE FOREVER BOUND,♪हमेशाके लिएबाध्यहोने,-
...,...,...,...
1995,Do they not see that Allah enlarges the provis...,क्या उन लोगों ने (इतना भी) ग़ौर नहीं किया कि खु...,-
1996,Say: 'Who is the Lord of the heavens and the e...,"कहो, ""आकाशों और धरती का रब कौन है?"" कहो, ""अल्ल...",-
1997,-(CELL PHONE RINGING),(सेल फोन बज),D.
1998,"Yeah, smoke it up there, uh, Skippy.","- हाँ, धूंए में उड़ा दो, मेरे शेर.",.


In [76]:
p = [i for i in Path('./Inference/').glob("*FT*")]

In [79]:
df = pd.read_csv(p[0])
df

,Unnamed: 0.1,Unnamed: 0,Source_Dialogue,Reference,MT
0,0,0,"Give shots of injections or pills, but he must...",सुई लगाओ या गोली खिलाओ लेकिन इसे जल्दी से ठीक ...,NaN
1,1,1,"They said, “O Shuaib, we do not understand muc...",और वह लोग कहने लगे ऐ शुएब जो बाते तुम कहते हो ...,.
2,2,2,- Yeah.,- हाँ.,D.
3,3,3,If evil befalls him he is perturbed;,"जि उसे तकलीफ़ पहुँचती है तो घबरा उठता है,",NaN
4,4,4,♪ BE FOREVER BOUND,♪हमेशाके लिएबाध्यहोने,NaN
...,...,...,...,...,...
1995,1995,1995,Do they not see that Allah enlarges the provis...,क्या उन लोगों ने (इतना भी) ग़ौर नहीं किया कि खु...,NaN
1996,1996,1996,Say: 'Who is the Lord of the heavens and the e...,"कहो, ""आकाशों और धरती का रब कौन है?"" कहो, ""अल्ल...",NaN
1997,1997,1997,-(CELL PHONE RINGING),(सेल फोन बज),D.
1998,1998,1998,"Yeah, smoke it up there, uh, Skippy.","- हाँ, धूंए में उड़ा दो, मेरे शेर.",.


In [80]:
for i in p:
    df = pd.read_csv(i)
    df.fillna('-')
    df.to_csv(i, sep='\t')

In [83]:
d1 = load_from_disk('../Layerwise/Data/ai4bharat/IN22-Conv/')['test']
d2 = load_from_disk('../Layerwise/Data/ai4bharat/IN22-Gen/')['test']
d3 = load_from_disk('../Layerwise/Data/openlanguagedata/flores_plus/')

In [84]:
d1

Dataset({
    features: ['doc_id', 'sent_id', 'topic', 'domain', 'prompt', 'scenario', 'speaker', 'turn', 'asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'eng_Latn', 'gom_Deva', 'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'mai_Deva', 'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya', 'pan_Guru', 'san_Deva', 'sat_Olck', 'snd_Deva', 'tam_Taml', 'tel_Telu', 'urd_Arab'],
    num_rows: 1503
})

In [85]:
d2

Dataset({
    features: ['context', 'source', 'url', 'domain', 'num_words', 'bucket', 'asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'eng_Latn', 'gom_Deva', 'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'mai_Deva', 'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya', 'pan_Guru', 'san_Deva', 'sat_Olck', 'snd_Deva', 'tam_Taml', 'tel_Telu', 'urd_Arab'],
    num_rows: 1024
})

In [86]:
d3

DatasetDict({
    dev: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 223328
    })
    devtest: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 220616
    })
})

In [87]:
traindata = load_dataset(
       "csv",
    data_files='/home/vish/yuvraj/Layerwise/Data/train_dataset.tsv',
    delimiter="\t",
    split="train"
    )

In [88]:
traindata

Dataset({
    features: ['Unnamed: 0', 'prompt'],
    num_rows: 3539
})

In [89]:
d = load_from_disk('../Layerwise/Data/Opus_Test_Data_en_hi/')

In [90]:
d

Dataset({
    features: ['translation', 'prompt', 'source', 'target'],
    num_rows: 2000
})

In [91]:
df = pd.read_csv('./final_system_scores.tsv', sep='\t')

In [92]:
df

,File_Name,BLEU,chrF++,BERTScore_F1,COMET,COMTAIL
0,llama_1B_INST_FT_sparseGPT_pruned_structured_7...,0.0036,0.1983,0.5841,0.3294,0.0953
1,llama_1B_INST_FT_sparseGPT_pruned_structured_5...,0.3957,1.0041,0.6440,0.3787,0.2173
2,llama_1B_INST_FT_sparseGPT_pruned_structured_2...,1.5020,10.4548,0.6998,0.4370,0.3395
3,llama_1B_INST_FT_sparseGPT_pruned_structured_5...,0.6342,1.1432,0.6347,0.3711,0.2009
4,llama_1B_INST_FT_sparseGPT_pruned_unstructured...,0.0094,0.3655,0.5980,0.2748,0.1336
5,llama_1B_INST_FT_sparseGPT_pruned_structured_2...,1.8947,12.4390,0.7160,0.4707,0.3855
6,llama_1B_INST_FT_sparseGPT_pruned_unstructured...,0.5466,3.9668,0.6554,0.3886,0.2245
7,llama_1B_INST_FT_sparseGPT_pruned_structured_7...,0.0029,0.1954,0.5377,0.2322,0.0983


In [93]:
import matplotlib.pyplot as plt
import seaborn as sns